## **Étape 1 : Préparation de l’environnement et ingestion**

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark=(
    SparkSession.builder
                .appName("E-shop")
                .master("spark://spark-master:7077")
                .getOrCreate()
)

sc=spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 20:04:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
print("Vesrion de Spark: ",spark.version)
print("Spark Master: ",sc.master)
print("Identifiant de l'application: ",sc.applicationId)

Vesrion de Spark:  3.5.1
Spark Master:  spark://spark-master:7077
Identifiant de l'application:  app-20261113200405-0000


In [5]:
from pyspark.sql.types import StructField, StringType, StructType, DateType, IntegerType, DecimalType

4. 

In [6]:
BASE_DIR="/home/jovyan/work"
SOURCE_DIR=f"{BASE_DIR}/data/source"

In [7]:
def string_schema(cols):
    return StructType([StructField(c,StringType(), True) for c in cols])

read_schemas={
    "customers": string_schema(["customer_id", "customer_name", "region", "registration_date"]),
    "products": string_schema(["product_id", "product_name", "category", "unit_price", "unit_cost"]),
    "orders":      string_schema(["order_id", "customer_id", "order_date", "status", "promotion_code"]),
    "order_items": string_schema(["order_item_id", "order_id", "product_id", "quantity", "unit_price"]),
}

def read_csv(name):
    return(
        spark.read
        .option("header",True)
        .option("sep",';')
        .schema(read_schemas[name])
        .csv(f"{SOURCE_DIR}/{name}.csv")
    )

In [8]:

customers_df= read_csv("customers")
orders_df= read_csv("orders")
order_items_df= read_csv("order_items")
products_df= read_csv("products")

promotions_df= spark.read.text(f"{SOURCE_DIR}/promotions.jsonl")

5. 

In [9]:

customers_expected = StructType([
    StructField("customer_id",       StringType(), False),
    StructField("customer_name",     StringType(), False),
    StructField("region",            StringType(), False),
    StructField("registration_date", DateType(),   False),
])

products_expected = StructType([
    StructField("product_id",   StringType(),      False),
    StructField("product_name", StringType(),      False),
    StructField("category",     StringType(),      False),
    StructField("unit_price",   DecimalType(10,2), False),  # prix catalogue
    StructField("unit_cost",    DecimalType(10,2), False),  # coût unitaire
])

orders_expected = StructType([
    StructField("order_id",       StringType(), False),
    StructField("customer_id",    StringType(), False),
    StructField("order_date",     DateType(),   False),
    StructField("status",         StringType(), False),
    StructField("promotion_code", StringType(), True),      # facultatif
])

order_items_expected = StructType([
    StructField("order_item_id", StringType(),      False),
    StructField("order_id",      StringType(),      False),
    StructField("product_id",    StringType(),      False),
    StructField("quantity",      IntegerType(),     False),
    StructField("unit_price",    DecimalType(10,2), False),  # prix transactionnel
])

promotions_expected = StructType([
    StructField("promotion_code", StringType(),      False),
    StructField("discount_rate",  DecimalType(5,4),  False),
    StructField("start_date",     DateType(),        False),
    StructField("end_date",       DateType(),        False),
])

expected_schemas = {
    "customers": customers_expected, "products": products_expected,
    "orders": orders_expected, "order_items": order_items_expected,
    "promotions": promotions_expected,
}

for name, schema in expected_schemas.items():
    print(f"--- {name} ---")
    for f in schema.fields:
        print(f"  {f.name:18} {f.dataType.simpleString():15} nullable={f.nullable}")

--- customers ---
  customer_id        string          nullable=False
  customer_name      string          nullable=False
  region             string          nullable=False
  registration_date  date            nullable=False
--- products ---
  product_id         string          nullable=False
  product_name       string          nullable=False
  category           string          nullable=False
  unit_price         decimal(10,2)   nullable=False
  unit_cost          decimal(10,2)   nullable=False
--- orders ---
  order_id           string          nullable=False
  customer_id        string          nullable=False
  order_date         date            nullable=False
  status             string          nullable=False
  promotion_code     string          nullable=True
--- order_items ---
  order_item_id      string          nullable=False
  order_id           string          nullable=False
  product_id         string          nullable=False
  quantity           int             nullable=F

#### **Vérification des volumes**

In [10]:
dfs = {
    "customers": customers_df, "products": products_df, "orders": orders_df,
    "order_items": order_items_df, "promotions": promotions_df,
}
expected = {"customers": 32, "products": 14, "orders": 44, "order_items": 86, "promotions": 6}

for name, df in dfs.items():
    print(f"{name:12} lu={df.count():3}  attendu={expected[name]}")
    df.printSchema()
    df.show(6, truncate=False)

customers    lu= 32  attendu=32
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
|C006       |Amine 06     |Nord  |2024-02-12       |
+-----------+-------------+------+-----------------+
only showing top 6 rows

products     lu= 14  attendu=14
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- unit_cost: string (nul

promotions   lu=  6  attendu=6
root
 |-- value: string (nullable = true)

+----------------------------------------------------------------------------------------------------+
|value                                                                                               |
+----------------------------------------------------------------------------------------------------+
|{"promotion_code":"PROMO5","discount_rate":0.05,"start_date":"2025-01-01","end_date":"2025-12-31"}  |
|{"promotion_code":"PROMO10","discount_rate":0.1,"start_date":"2025-05-01","end_date":"2025-05-31"}  |
|{"promotion_code":"SUMMER15","discount_rate":0.15,"start_date":"2025-06-01","end_date":"2025-08-31"}|
|{"promotion_code":"VIP20","discount_rate":0.2,"start_date":"2025-01-01","end_date":"2025-12-31"}    |
|{"promotion_code":"OLD30","discount_rate":0.3,"start_date":"2024-01-01","end_date":"2024-12-31"}    |
|{"promotion_code":"BAD_JSON","discount_rate":0.10,"start_date":"2025-06-01","end_date":             |

8. 
* **Valeur vide (null)**:
Une case vide dans le CSV devient null dans Spark.

    * O0001;C007;2025-05-04;PAID; : le code promo est vide. Pas une erreur : la commande n’a simplement pas de promotion.
    * C031;;Nord;2025-02-30 : le nom est vide. C’est une erreur, car un client doit avoir un nom.

* **Conversion numérique**:
On veut transformer la quantité (texte) en nombre entier.

    * I9004;...;abc;45.00 : abc n’est pas un nombre, donc la conversion donne null.
    * I9003 (quantité -2) et I9005 (quantité 0) : la conversion réussit, mais la valeur n’a pas de sens pour une quantité.

* **Date invalide**:
    * 2025-02-30 n’existe pas (février n’a pas 30 jours). Spark ne plante pas forcément : il peut renvoyer null.

#### **Questions d’analyse** 


* Contrôler les schémas avant les transformations permet de connaître le type de chaque colonne et donc les opérations autorisées (calculs sur les nombres, comparaisons sur les dates), de détecter tôt les colonnes manquantes ou mal typées, d’éviter les conversions silencieuses qui produisent des null et faussent les agrégations, et de garantir des calculs financiers exacts et reproductibles.

* Une lecture réussie signifie seulement que Spark a pu ouvrir et lire le fichier sans erreur technique, sans rien dire sur la validité du contenu : des valeurs nulles, des conversions impossibles, des doublons ou des dates invalides peuvent passer inaperçus. Une ingestion de qualité ajoute des contrôles (schéma, volumes, champs obligatoires, conversions, doublons) et de la traçabilité (manifeste, run_id), ce qui permet de détecter et documenter les anomalies avant de les traiter, et de pouvoir faire confiance aux données utilisées ensuite.

* Les données brutes doivent rester disponibles car elles constituent la référence immuable du pipeline. Elles permettent de rejouer les traitements après une correction de règle, d’auditer et de retracer chaque anomalie avec sa valeur d’origine, de vérifier qu’aucune ligne n’a été perdue entre les zones, et de faire évoluer les règles métier sans nouvelle collecte. Les supprimer ou les corriger dès l’ingestion rendrait impossible toute vérification ou reprise.

* Lire d’abord les champs CSV en chaînes de caractères préserve les valeurs originales et évite les conversions silencieuses : avec une lecture typée directe, une valeur invalide comme abc ou une date impossible comme 2025-02-30 devient null, ce qui la rend indiscernable d’une valeur simplement manquante. En convertissant ensuite explicitement, on peut comparer la valeur d’origine au résultat du cast, identifier précisément le motif d’erreur (manquant, non convertible, hors règle métier), conserver la valeur brute pour la quarantaine et l’audit, et garantir qu’aucune ligne n’est perdue à la lecture.

## **Étape 2 : Zone Bronze**

In [11]:
import os, json, uuid, shutil, json
from datetime import datetime

In [12]:

BRONZE_RAW=f"{BASE_DIR}/data_lakehouse/bronze/raw"
BRONZE_PQ=f"{BASE_DIR}/data_lakehouse/bronze/parquet"
MANIFEST_DIR=f"{BASE_DIR}/data_lakehouse/metadata/manifests"

run_id = str(uuid.uuid4())
os.makedirs(BRONZE_RAW, exist_ok=True)
os.makedirs(BRONZE_PQ, exist_ok=True)
os.makedirs(f"{MANIFEST_DIR}/{run_id}", exist_ok=True)

sources = {
    "customers":   "customers.csv",
    "products":    "products.csv",
    "orders":      "orders.csv",
    "order_items": "order_items.csv",
    "promotions":  "promotions.jsonl",
}

**-> ingestion raw (copie sans modification)**

In [13]:
raw_status={}

for name, filename in sources.items():
    try:
        shutil.copy2(f"{SOURCE_DIR}/{filename}",f"{BRONZE_RAW}/{filename}")
        raw_status[name]="SUCCESS"
    except Exception as e:
        raw_status[name]=f"FAILED: {e}"


**->ingestion Parquet (depuis la copie Bronze)**

In [14]:
rows_ingested= {}
for name, df in dfs.items():
    rows_ingested[name]=df.count()
    df.write.mode("overwrite").parquet(f"{BRONZE_PQ}/{name}")

print(rows_ingested)

{'customers': 32, 'products': 14, 'orders': 44, 'order_items': 86, 'promotions': 6}


**-> Ecrire les Manifests**

In [15]:
for name, filename in sources.items():
    relu= spark.read.parquet(f"{BRONZE_PQ}/{name}").count()
    ok=raw_status[name]=="SUCCESS" and relu==rows_ingested[name]

    manifest = {
        "run_id":              run_id,
        "source_name":         name,
        "rows_ingested":       rows_ingested[name],
        "ingestion_timestamp": datetime.now().isoformat(timespec="seconds"),
        "bronze_path":         f"{BRONZE_RAW}/{filename}",
        "status":              "SUCCESS" if ok else "FAILED",
    }

    with open(f"{MANIFEST_DIR}/{run_id}/manifest_{name}.json", "w", encoding="utf-8") as f:
        json.dump(manifest,f, indent=2, ensure_ascii=False)

#### **Questions**

1. **Pourquoi Bronze ne supprime pas les anomalies ?**
Bronze est la copie fidèle de ce qui a été reçu. Les anomalies (doublons, abc, 2025-02-30, ligne JSON mal formée) doivent y rester pour être auditées et traitées en Silver avec des règles explicites. Les supprimer rendrait impossible la vérification de la conservation des lignes.

2. **Comment Bronze facilite le rejeu ?**
Elle fournit un point de départ immuable, indépendant des sources. Après correction d’une règle, on relance Silver et Gold depuis Bronze, sans recharger les fichiers, et on obtient des résultats comparables.

3. **Quelles informations retracent une ingestion ?**
Celles du manifeste : nom de la source, nombre de lignes ingérées, date et heure, chemin Bronze, statut, et run_id 


4. **Fichier source vs Parquet ?**
Le fichier source est une copie exacte de l’original, qui sert de preuve. Le Parquet est une représentation recodée par Spark (binaire, colonnes, schéma imposé), pratique à exploiter mais c’est déjà une transformation. Il complète raw/ sans le remplacer.

## **Étape 3 : Contrôle de la qualité et gestion des anomalies**

#### **-> Customers**

In [16]:
from pyspark.sql import functions as F

**1. Charger la source**

In [17]:
customers_raw= (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .parquet(f"{BRONZE_PQ}/customers")
)
customers_raw.printSchema()
customers_raw.show(10,truncate=False)
print("Nombre de lignes :", customers_raw.count()) 


root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
|C006       |Amine 06     |Nord  |2024-02-12       |
|C007       |Alice 07     |Sud   |2024-02-19       |
|C008       |Youssef 08   |Centre|2024-02-26       |
|C009       |Sara 09      |Est   |2024-03-04       |
|C010       |Omar 10      |Ouest |2024-03-11       |
+-----------+-------------+------+-----------------+
only showing top 10 rows

Nombre de lignes : 32


**-> Supprimer les éspaces dans les columns**

In [18]:
customers_raw=customers_raw.toDF(*[c.strip() for c in customers_raw.columns])

customers_clean= customers_raw.select(
    [F.trim(F.col(c)).alias(c) for c in customers_raw.columns]
)

**2. Vérifier les champs obligatoires**

In [19]:
null_values={}

customers_clean.select(
    F.sum(
        F.when(
            F.col("customer_id").isNull() | (F.trim(F.col("customer_id"))==""),
            1
        ).otherwise(0)
    ).alias("missing_ids"),

    F.sum(
        F.when(
            F.col("customer_name").isNull() | (F.trim(F.col("customer_name"))==""),
            1
        ).otherwise(0)
    ).alias("missing_names"),
    F.sum(
            F.when(
                F.col("region").isNull() | (F.trim(F.col("region"))==""),
                1
            ).otherwise(0)
        ).alias("missing_regions")
).show()


+-----------+-------------+---------------+
|missing_ids|missing_names|missing_regions|
+-----------+-------------+---------------+
|          0|            1|              0|
+-----------+-------------+---------------+



**3. Contrôler les dates**

In [20]:
customers_typed= customers_clean.withColumn(
    "registration_date_parsed",
    F.to_date("registration_date","yyyy-MM-dd")
)

customers_typed.select(
    "customer_id",
    "registration_date",
    "registration_date_parsed"
).show(1000,truncate=False)

+-----------+-----------------+------------------------+
|customer_id|registration_date|registration_date_parsed|
+-----------+-----------------+------------------------+
|C001       |2024-01-08       |2024-01-08              |
|C002       |2024-01-15       |2024-01-15              |
|C003       |2024-01-22       |2024-01-22              |
|C004       |2024-01-29       |2024-01-29              |
|C005       |2024-02-05       |2024-02-05              |
|C006       |2024-02-12       |2024-02-12              |
|C007       |2024-02-19       |2024-02-19              |
|C008       |2024-02-26       |2024-02-26              |
|C009       |2024-03-04       |2024-03-04              |
|C010       |2024-03-11       |2024-03-11              |
|C011       |2024-03-18       |2024-03-18              |
|C012       |2024-03-25       |2024-03-25              |
|C013       |2024-04-01       |2024-04-01              |
|C014       |2024-04-08       |2024-04-08              |
|C015       |2024-04-15       |

**la dernière ligne a echoué (de customer_id : C031), car il egale à 2025-02-30 et Spark la convertit en NULL**

**4. Détecter les doublons**

In [21]:
duplicate_ids = ( 
customers_raw 
.groupBy("customer_id") 
.count() 
.filter(F.col("count") > 1) 
) 
duplicate_ids.show(truncate=False) 



+-----------+-----+
|customer_id|count|
+-----------+-----+
|C005       |2    |
+-----------+-----+



**-> Ces doublons sont strictement identiquent, alors on doit garder une et placer l'autre dans le dossier audit**

**-> 

**5. Construire une justification des anomalies**

In [22]:
customers_checked = customers_typed.withColumn(
    "quality_reason",
    F.concat_ws(
        " | ",
        F.when(
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id"))==""),
            F.lit("ID_MISSING")
        ),
        F.when( 
            F.col("customer_name").isNull() 
            | (F.trim(F.col("customer_name")) == ""), 
            F.lit("NOM_MANQUANT") 
        ), 
        F.when( 
            F.col("region").isNull() 
            | (F.trim(F.col("region")) == ""), 
            F.lit("REGION_MANQUANTE") 
        ), 
        F.when( 
            F.col("registration_date").isNotNull() 
            & (F.trim(F.col("registration_date")) != "") 
            & F.col("registration_date_parsed").isNull(), 
            F.lit("DATE_INVALIDE") 
        ) 
    )
)
customers_checked.show(2000,truncate=False)

+-----------+-------------+------+-----------------+------------------------+----------------------------+
|customer_id|customer_name|region|registration_date|registration_date_parsed|quality_reason              |
+-----------+-------------+------+-----------------+------------------------+----------------------------+
|C001       |Alice 01     |Nord  |2024-01-08       |2024-01-08              |                            |
|C002       |Youssef 02   |Sud   |2024-01-15       |2024-01-15              |                            |
|C003       |Sara 03      |Centre|2024-01-22       |2024-01-22              |                            |
|C004       |Omar 04      |Est   |2024-01-29       |2024-01-29              |                            |
|C005       |Nadia 05     |Ouest |2024-02-05       |2024-02-05              |                            |
|C006       |Amine 06     |Nord  |2024-02-12       |2024-02-12              |                            |
|C007       |Alice 07     |Sud   |202

**6. Séparer les données et vérifier leur conservation** 


In [23]:
customers_valid =(customers_checked
                  .filter(
                        F.col("quality_reason").isNull() 
                        | (F.trim(F.col("quality_reason")) == "") 
                    )
                    .dropDuplicates()
                    .drop("registration_date","quality_reason")
                    .withColumnRenamed("registration_date_parsed","registration_date")
)

customers_valid.show(2000,truncate=False)



+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C020       |Youssef 20   |Ouest |2024-05-20       |
|C028       |Omar 28      |Centre|2024-07-15       |
|C011       |Nadia 11     |Nord  |2024-03-18       |
|C008       |Youssef 08   |Centre|2024-02-26       |
|C015       |Sara 15      |Ouest |2024-04-15       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C012       |Amine 12     |Sud   |2024-03-25       |
|C016       |Omar 16      |Nord  |2024-04-22       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
|C009       |Sara 09      |Est   |2024-03-04       |
|C007       |Alice 07     |Sud   |2024-02-19       |
|C006       |Amine 06     |Nord  |2024-02-12       |
|C029       |Nadia 29     |Est   |2024-07-22       |
|C030       |Amine 30     |Ouest |2024-07-29       |
|C010       |Omar 10      |Ouest |2024-03-11       |
|C017       |Nadia 17     |Sud   |2024-04-29  

In [24]:
customers_quarantine = customers_checked.filter(
    F.col("quality_reason").isNotNull() 
    & (F.trim(F.col("quality_reason")) != "") 
)

customers_quarantine.show(2000,truncate=False)

+-----------+-------------+------+-----------------+------------------------+----------------------------+
|customer_id|customer_name|region|registration_date|registration_date_parsed|quality_reason              |
+-----------+-------------+------+-----------------+------------------------+----------------------------+
|C031       |NULL         |Nord  |2025-02-30       |NULL                    |NOM_MANQUANT | DATE_INVALIDE|
+-----------+-------------+------+-----------------+------------------------+----------------------------+



In [25]:
customers_cols= customers_typed.columns

customers_duplications=(
    customers_typed
    .groupBy(customers_cols)
    .agg(F.count("*").alias("occurrences"))
    .filter("occurrences > 1")
    .withColumn("copie_key",F.col("customer_id"))
    .withColumn("extra_copies", F.col("occurrences")-1)
    .withColumn("run_id", F.lit(run_id))
)

customers_duplications.show(truncate=False)

+-----------+-------------+------+-----------------+------------------------+-----------+---------+------------+------------------------------------+
|customer_id|customer_name|region|registration_date|registration_date_parsed|occurrences|copie_key|extra_copies|run_id                              |
+-----------+-------------+------+-----------------+------------------------+-----------+---------+------------+------------------------------------+
|C005       |Nadia 05     |Ouest |2024-02-05       |2024-02-05              |2          |C005     |1           |f206b212-0ef8-41db-9f51-dc601a6785dd|
+-----------+-------------+------+-----------------+------------------------+-----------+---------+------------+------------------------------------+



**Canonical Record Verification for valid data**

In [26]:
dup_ids = (customers_valid
           .groupBy("customer_id")
           .count()
           .filter(F.col("count")>1)

)
dup_ids.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



**Apply the rool: *Nombre initial de lignes = nombre d'enregistrements canoniques valides + nombre d'enregistrements en quarantaine + nombre de copies supplémentaires de doublons exacts tracées.*** 

In [27]:
initial_rows=customers_raw.count()

valid_rows=customers_valid.count()
quarantine_rows=customers_quarantine.count()
dup_count=customers_duplications.count()

print(initial_rows==valid_rows+quarantine_rows+dup_count)


True


**7. Spark Silver**

In [28]:
SILVER_DIR=f"{BASE_DIR}/data_lakehouse/silver"

customers_valid.write.mode("overwrite").parquet(f"{SILVER_DIR}/customers")

In [29]:
QUAR=f"{BASE_DIR}/data_lakehouse/quarantine"

customers_quarantine= (
    customers_quarantine
    .withColumn("run_id", F.lit(run_id))
    .withColumn("quarantined_at", F.current_timestamp())
    .drop("registration_date_parsed")
)

customers_quarantine.write.mode("overwrite").parquet(f"{QUAR}/customers")

In [30]:
AUDIT_DIR=f"{BASE_DIR}/data_lakehouse/metadata/audit"

customers_duplications.write.mode("overwrite").parquet(f"{AUDIT_DIR}/customers_duplicates")

#### **-> Products**

In [31]:
products_raw = (
    spark.read
    .option("header",True)
    .option("inferSchema",False)
    .parquet(f"{BRONZE_PQ}/products")
)
products_raw.show(200,truncate=False)
products_raw.printSchema()
print("Nombre de lignes: ", products_raw.count())

+----------+----------------+------------+----------+---------+
|product_id|product_name    |category    |unit_price|unit_cost|
+----------+----------------+------------+----------+---------+
|P001      |Laptop          |Informatique|800.00    |600.00   |
|P002      |Mouse           |Accessoires |20.00     |8.00     |
|P003      |Screen          |Informatique|200.00    |140.00   |
|P004      |Keyboard        |Accessoires |45.00     |25.00    |
|P005      |Tablet          |Informatique|320.00    |230.00   |
|P006      |Headphones      |Audio       |75.00     |40.00    |
|P007      |Webcam          |Accessoires |55.00     |30.00    |
|P008      |Printer         |Bureau      |180.00    |125.00   |
|P009      |SSD 1TB         |Stockage    |95.00     |58.00    |
|P010      |Router          |Réseau      |110.00    |70.00    |
|P011      |Microphone      |Audio       |65.00     |35.00    |
|P012      |USB Hub         |Accessoires |25.00     |11.00    |
|P003      |Screen          |Informatiqu

**2.**

In [32]:
products_raw.select(
    F.sum(
        F.when(
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id"))=="")
            , 1
        ).otherwise(0)
    ).alias("id_manquant"),

    F.sum(
        F.when(
            F.col("product_name").isNull()
            | (F.trim(F.col("product_name"))=="")
            , 1
        ).otherwise(0)
    ).alias("nom_manquant"),

    F.sum(
            F.when(
                F.col("category").isNull()
                | (F.trim(F.col("category"))=="")
                , 1
            ).otherwise(0)
        ).alias("category_manquant"),

    F.sum(
        F.when(
            F.col("unit_price").isNull()
            | (F.trim(F.col("unit_price"))=="")
            , 1
        ).otherwise(0)
    ).alias("unit_price_manquant"),

    F.sum(
        F.when(
            F.col("unit_cost").isNull()
            | (F.trim(F.col("unit_cost"))=="")
            , 1
        ).otherwise(0)
    ).alias("unit_cost_manquant")  

).show()

+-----------+------------+-----------------+-------------------+------------------+
|id_manquant|nom_manquant|category_manquant|unit_price_manquant|unit_cost_manquant|
+-----------+------------+-----------------+-------------------+------------------+
|          0|           0|                0|                  0|                 0|
+-----------+------------+-----------------+-------------------+------------------+



**unique values for product_name and category**

In [33]:
products_raw.select("product_name").distinct().show()
products_raw.select("category").distinct().show()

+----------------+
|    product_name|
+----------------+
|         SSD 1TB|
|          Router|
|          Webcam|
|         USB Hub|
|          Laptop|
|           Mouse|
|      Microphone|
|          Tablet|
|         Printer|
|        Keyboard|
|          Screen|
|Broken Cost Item|
|      Headphones|
+----------------+



+------------+
|    category|
+------------+
|Informatique|
|      Bureau|
| Accessoires|
|    Stockage|
|      Divers|
|       Audio|
|      Réseau|
+------------+



**Types unit_price et unit_cost to double**

In [34]:
from pyspark.sql.types import DecimalType

In [35]:
products_typed = products_raw.withColumn(
    "unit_price_parsed",
    F.col("unit_price").cast(DecimalType(10,2))
)

products_typed = products_typed.withColumn(
    "unit_cost_parsed",
    F.col("unit_cost").cast(DecimalType(10,2))
)

products_typed.show(20)

+----------+----------------+------------+----------+---------+-----------------+----------------+
|product_id|    product_name|    category|unit_price|unit_cost|unit_price_parsed|unit_cost_parsed|
+----------+----------------+------------+----------+---------+-----------------+----------------+
|      P001|          Laptop|Informatique|    800.00|   600.00|           800.00|          600.00|
|      P002|           Mouse| Accessoires|     20.00|     8.00|            20.00|            8.00|
|      P003|          Screen|Informatique|    200.00|   140.00|           200.00|          140.00|
|      P004|        Keyboard| Accessoires|     45.00|    25.00|            45.00|           25.00|
|      P005|          Tablet|Informatique|    320.00|   230.00|           320.00|          230.00|
|      P006|      Headphones|       Audio|     75.00|    40.00|            75.00|           40.00|
|      P007|          Webcam| Accessoires|     55.00|    30.00|            55.00|           30.00|
|      P00

**-> Identifier les prix negatif**

In [36]:
neg_price_cost = (
    products_typed
    .filter(
        (F.col("unit_price")<0) | (F.col("unit_cost")<0)
    )
).show()

+----------+----------------+--------+----------+---------+-----------------+----------------+
|product_id|    product_name|category|unit_price|unit_cost|unit_price_parsed|unit_cost_parsed|
+----------+----------------+--------+----------+---------+-----------------+----------------+
|      P013|Broken Cost Item|  Divers|     30.00|    -4.00|            30.00|           -4.00|
+----------+----------------+--------+----------+---------+-----------------+----------------+



**Duplications detection**

In [37]:
duplicated_ids = (
    products_typed
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
).show()

+----------+-----+
|product_id|count|
+----------+-----+
|      P003|    2|
+----------+-----+



In [38]:
products_cols= products_typed.columns

products_duplications = (
    products_typed
    .groupBy(products_cols)
    .agg(
        F.count("*").alias("occurences")
    )
    .filter(F.col("occurences") > 1)
    .withColumn("copie_key",F.col("product_id"))
    .withColumn("extra_copies", F.col("occurences")-1)
    .withColumn("run_id", F.lit(run_id))
)

products_duplications.show()


+----------+------------+------------+----------+---------+-----------------+----------------+----------+---------+------------+--------------------+
|product_id|product_name|    category|unit_price|unit_cost|unit_price_parsed|unit_cost_parsed|occurences|copie_key|extra_copies|              run_id|
+----------+------------+------------+----------+---------+-----------------+----------------+----------+---------+------------+--------------------+
|      P003|      Screen|Informatique|    200.00|   140.00|           200.00|          140.00|         2|     P003|           1|f206b212-0ef8-41d...|
+----------+------------+------------+----------+---------+-----------------+----------------+----------+---------+------------+--------------------+



In [39]:
products_checked = products_typed.withColumn(
    "quality_reason",
    F.concat_ws( 
        " | ", 
        F.when( 
            F.col("product_id").isNull() 
            | (F.trim(F.col("product_id")) == ""), 
            F.lit("ID_MANQUANT") 
        ), 
        F.when( 
            F.col("product_name").isNull() 
            | (F.trim(F.col("product_name")) == ""), 
            F.lit("NOM_MANQUANT") 
        ), 
        F.when( 
            F.col("category").isNull() 
            | (F.trim(F.col("category")) == ""), 
            F.lit("CATEGORY_MANQUANTE") 
        ), 
        F.when( 
            F.col("unit_price") < 0,
            F.lit("UNIT_PRICE_NEGATIVE")
        ),
        F.when( 
            F.col("unit_cost") < 0,
            F.lit("UNIT_COST_NEGATIVE")
        )
    )
)
products_checked.show()

+----------+----------------+------------+----------+---------+-----------------+----------------+------------------+
|product_id|    product_name|    category|unit_price|unit_cost|unit_price_parsed|unit_cost_parsed|    quality_reason|
+----------+----------------+------------+----------+---------+-----------------+----------------+------------------+
|      P001|          Laptop|Informatique|    800.00|   600.00|           800.00|          600.00|                  |
|      P002|           Mouse| Accessoires|     20.00|     8.00|            20.00|            8.00|                  |
|      P003|          Screen|Informatique|    200.00|   140.00|           200.00|          140.00|                  |
|      P004|        Keyboard| Accessoires|     45.00|    25.00|            45.00|           25.00|                  |
|      P005|          Tablet|Informatique|    320.00|   230.00|           320.00|          230.00|                  |
|      P006|      Headphones|       Audio|     75.00|   

**-> Products valide**

In [40]:
products_valid = (
    products_checked
    .filter(
        (F.trim(F.col("quality_reason"))=="")
    )
    .dropDuplicates()
    .drop("unit_price","unit_cost","quality_reason")
    .withColumnRenamed("unit_price_parsed","unit_catalog_price")
    .withColumnRenamed("unit_cost_parsed","unit_cost")
)

products_valid.show(20, truncate=False)

+----------+------------+------------+------------------+---------+
|product_id|product_name|category    |unit_catalog_price|unit_cost|
+----------+------------+------------+------------------+---------+
|P011      |Microphone  |Audio       |65.00             |35.00    |
|P001      |Laptop      |Informatique|800.00            |600.00   |
|P010      |Router      |Réseau      |110.00            |70.00    |
|P005      |Tablet      |Informatique|320.00            |230.00   |
|P009      |SSD 1TB     |Stockage    |95.00             |58.00    |
|P012      |USB Hub     |Accessoires |25.00             |11.00    |
|P006      |Headphones  |Audio       |75.00             |40.00    |
|P002      |Mouse       |Accessoires |20.00             |8.00     |
|P003      |Screen      |Informatique|200.00            |140.00   |
|P007      |Webcam      |Accessoires |55.00             |30.00    |
|P004      |Keyboard    |Accessoires |45.00             |25.00    |
|P008      |Printer     |Bureau      |180.00    

In [41]:
products_quarantine = (
    products_checked.filter(
        (F.trim(F.col("quality_reason"))!="")
    )
    .drop("unit_price_parsed","unit_cost_parsed")
)

products_quarantine.show()

+----------+----------------+--------+----------+---------+------------------+
|product_id|    product_name|category|unit_price|unit_cost|    quality_reason|
+----------+----------------+--------+----------+---------+------------------+
|      P013|Broken Cost Item|  Divers|     30.00|    -4.00|UNIT_COST_NEGATIVE|
+----------+----------------+--------+----------+---------+------------------+



**-> Enregistrer en Silver**

In [42]:
products_valid.write.mode("overwrite").parquet(f"{SILVER_DIR}/products")

products_quarantine.write.mode("overwrite").parquet(f"{QUAR}/products")

products_duplications.write.mode("overwrite").parquet(f"{AUDIT_DIR}/products_duplicates")

### **-> Promotions**

In [43]:
promotions_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .parquet(f"{BRONZE_PQ}/promotions")
)

promotions_raw.show(5, truncate=False)
promotions_raw.printSchema()
print("nombre de lignes: ",promotions_raw.count())

+----------------------------------------------------------------------------------------------------+
|value                                                                                               |
+----------------------------------------------------------------------------------------------------+
|{"promotion_code":"PROMO5","discount_rate":0.05,"start_date":"2025-01-01","end_date":"2025-12-31"}  |
|{"promotion_code":"PROMO10","discount_rate":0.1,"start_date":"2025-05-01","end_date":"2025-05-31"}  |
|{"promotion_code":"SUMMER15","discount_rate":0.15,"start_date":"2025-06-01","end_date":"2025-08-31"}|
|{"promotion_code":"VIP20","discount_rate":0.2,"start_date":"2025-01-01","end_date":"2025-12-31"}    |
|{"promotion_code":"OLD30","discount_rate":0.3,"start_date":"2024-01-01","end_date":"2024-12-31"}    |
+----------------------------------------------------------------------------------------------------+
only showing top 5 rows

root
 |-- value: string (nullable = true)

nombr

**-> Le parsing de schema de promotions**

In [44]:
promo_schema = StructType([
    StructField("promotion_code", StringType()),
    StructField("discount_rate", StringType()),
    StructField("start_date", StringType()),
    StructField("end_date", StringType()),
])

promotions_parsed =(
    promotions_raw
    .withColumnRenamed("value","raw_line")
    .withColumn("p", F.from_json("raw_line" , promo_schema))
    .withColumn("parsing_ok",
                F.col("p").isNotNull()
                & F.col("p.promotion_code").isNotNull()
                & F.col("p.end_date").isNotNull()
            )
)

promotions_parsed.select(
    "raw_line",
    F.col("p").isNotNull().alias("p_not_null"),
    "parsing_ok",
).show(truncate=False)

+----------------------------------------------------------------------------------------------------+----------+----------+
|raw_line                                                                                            |p_not_null|parsing_ok|
+----------------------------------------------------------------------------------------------------+----------+----------+
|{"promotion_code":"PROMO5","discount_rate":0.05,"start_date":"2025-01-01","end_date":"2025-12-31"}  |true      |true      |
|{"promotion_code":"PROMO10","discount_rate":0.1,"start_date":"2025-05-01","end_date":"2025-05-31"}  |true      |true      |
|{"promotion_code":"SUMMER15","discount_rate":0.15,"start_date":"2025-06-01","end_date":"2025-08-31"}|true      |true      |
|{"promotion_code":"VIP20","discount_rate":0.2,"start_date":"2025-01-01","end_date":"2025-12-31"}    |true      |true      |
|{"promotion_code":"OLD30","discount_rate":0.3,"start_date":"2024-01-01","end_date":"2024-12-31"}    |true      |true      |


In [45]:
promotions_typed = (
    promotions_parsed
    .select(
        "raw_line",
        "parsing_ok",
        F.col("p.promotion_code").alias("promotion_code"),
        F.col("p.discount_rate").alias("discount_rate"),
        F.col("p.start_date").alias("start_date"),
        F.col("p.end_date").alias("end_date"),
        F.col("discount_rate").cast("decimal(5,4)").alias("discount_rate_casted"),
        F.to_date(F.trim("p.start_date"),"yyyy-MM-dd").alias("start_date_casted"),
        F.to_date(F.trim("p.end_date"),"yyyy-MM-dd").alias("end_date_casted"),
    )
)

promotions_typed.show(6,truncate=False)

+----------------------------------------------------------------------------------------------------+----------+--------------+-------------+----------+----------+--------------------+-----------------+---------------+
|raw_line                                                                                            |parsing_ok|promotion_code|discount_rate|start_date|end_date  |discount_rate_casted|start_date_casted|end_date_casted|
+----------------------------------------------------------------------------------------------------+----------+--------------+-------------+----------+----------+--------------------+-----------------+---------------+
|{"promotion_code":"PROMO5","discount_rate":0.05,"start_date":"2025-01-01","end_date":"2025-12-31"}  |true      |PROMO5        |0.05         |2025-01-01|2025-12-31|0.0500              |2025-01-01       |2025-12-31     |
|{"promotion_code":"PROMO10","discount_rate":0.1,"start_date":"2025-05-01","end_date":"2025-05-31"}  |true      |PROMO10

In [46]:
promotions_checked = promotions_typed.withColumn(
    "quality_reason",
    F.concat_ws(" | ",
        
        F.when(
                F.col("parsing_ok")=="false",
                F.lit("PARSING_FAILED")
       ),
        # code
        F.when(F.col("promotion_code").isNull() | (F.col("promotion_code") == ""),
               F.lit("CODE_MANQUANT")),

        # taux
        F.when(F.col("discount_rate").isNull() | (F.trim("discount_rate") == ""),
               F.lit("TAUX_MANQUANT")),
        F.when(F.col("discount_rate").isNotNull() & (F.trim("discount_rate") != "")
               & F.col("discount_rate_casted").isNull(),
               F.lit("TAUX_NON_NUMERIQUE")),
        F.when((F.col("discount_rate_casted") < 0) | (F.col("discount_rate_casted") > 1),
               F.lit("TAUX_HORS_INTERVALLE")),

        # dates
        F.when(F.col("start_date").isNull() | (F.trim("start_date") == ""),
               F.lit("DATE_DEBUT_MANQUANTE")),
        F.when(F.col("start_date").isNotNull() & (F.trim("start_date") != "")
               & F.col("start_date_casted").isNull(),
               F.lit("DATE_DEBUT_INVALIDE")),
       F.when(F.col("end_date").isNull() | (F.trim("end_date") == ""),
               F.lit("DATE_FIN_MANQUANTE")),
        F.when(F.col("end_date").isNotNull() & (F.trim("end_date") != "")
               & F.col("end_date_casted").isNull(),
               F.lit("DATE_FIN_INVALIDE")),
        F.when(F.col("start_date_casted").isNotNull() & F.col("end_date_casted").isNotNull()
               & (F.col("start_date_casted") > F.col("end_date_casted")),
               F.lit("PERIODE_INCOHERENTE")),
    )
)

promotions_checked.show(truncate=False)

+----------------------------------------------------------------------------------------------------+----------+--------------+-------------+----------+----------+--------------------+-----------------+---------------+------------------------------------------------------------------------------------------+
|raw_line                                                                                            |parsing_ok|promotion_code|discount_rate|start_date|end_date  |discount_rate_casted|start_date_casted|end_date_casted|quality_reason                                                                            |
+----------------------------------------------------------------------------------------------------+----------+--------------+-------------+----------+----------+--------------------+-----------------+---------------+------------------------------------------------------------------------------------------+
|{"promotion_code":"PROMO5","discount_rate":0.05,"start_date":"2025

In [150]:
promotions_valid= (
    promotions_checked
    .filter(
        (F.trim(F.col("quality_reason"))=="")
    )
    .drop("raw_line","parsing_ok","discount_rate","start_date","end_date","quality_reason")
    .withColumnRenamed("discount_rate_casted","discount_rate")
    .withColumnRenamed("start_date_casted","start_date")
    .withColumnRenamed("end_date_casted","end_date")
    .withColumn("discount_rate", F.col("discount_rate").cast("decimal(5,4)"))
)

promotions_valid.show()

+--------------+-------------+----------+----------+
|promotion_code|discount_rate|start_date|  end_date|
+--------------+-------------+----------+----------+
|        PROMO5|       0.0500|2025-01-01|2025-12-31|
|       PROMO10|       0.1000|2025-05-01|2025-05-31|
|      SUMMER15|       0.1500|2025-06-01|2025-08-31|
|         VIP20|       0.2000|2025-01-01|2025-12-31|
|         OLD30|       0.3000|2024-01-01|2024-12-31|
+--------------+-------------+----------+----------+



In [208]:
promotions_valid.printSchema()

root
 |-- promotion_code: string (nullable = true)
 |-- discount_rate: decimal(5,4) (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)



In [152]:
promotions_quarantine = (
    promotions_checked.filter(
        (F.trim(F.col("quality_reason"))!="")
    )
)

promotions_quarantine.show()

+--------------------+----------+--------------+-------------+----------+--------+--------------------+-----------------+---------------+--------------------+
|            raw_line|parsing_ok|promotion_code|discount_rate|start_date|end_date|discount_rate_casted|start_date_casted|end_date_casted|      quality_reason|
+--------------------+----------+--------------+-------------+----------+--------+--------------------+-----------------+---------------+--------------------+
|{"promotion_code"...|     false|          NULL|         NULL|      NULL|    NULL|                NULL|             NULL|           NULL|PARSING_FAILED | ...|
+--------------------+----------+--------------+-------------+----------+--------+--------------------+-----------------+---------------+--------------------+



In [153]:
promotions_valid.write.mode("overwrite").parquet(f"{SILVER_DIR}/promotions")
promotions_quarantine.write.mode("overwrite").parquet(f"{QUAR}/promotions")

#### **-> Orders**

In [154]:
orders_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema",False)
    .parquet(f"{BRONZE_PQ}/orders")
)

orders_raw.show(1000, truncate=False)
orders_raw.printSchema()
print("nombre de lignes: ", orders_raw.count())

+--------+-----------+----------+-----------+--------------+
|order_id|customer_id|order_date|status     |promotion_code|
+--------+-----------+----------+-----------+--------------+
|O0001   |C007       |2025-05-04|PAID       |              |
|O0002   |C014       |2025-05-07|DELIVERED  |PROMO5        |
|O0003   |C021       |2025-05-10|CANCELLED  |PROMO10       |
|O0004   |C028       |2025-05-13|PENDING    |SUMMER15      |
|O0005   |C005       |2025-05-16|PAID       |VIP20         |
|O0006   |C012       |2025-05-19|DELIVERED  |UNKNOWN       |
|O0007   |C019       |2025-05-22|CANCELLED  |OLD30         |
|O0008   |C026       |2025-05-25|PENDING    |              |
|O0009   |C003       |2025-05-28|PAID       |PROMO5        |
|O0010   |C010       |2025-05-31|DELIVERED  |PROMO10       |
|O0011   |C017       |2025-06-03|CANCELLED  |SUMMER15      |
|O0012   |C024       |2025-06-06|PENDING    |VIP20         |
|O0013   |C001       |2025-06-09|PAID       |UNKNOWN       |
|O0014   |C008       |20

**-> Let's see if there is any missing values**

In [155]:
orders_raw.select(
    F.sum(
        F.when(
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id"))==""),
            1
        ).otherwise(0)
    ).alias("id_manquant"),
    F.sum(
        F.when(
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id"))==""),
            1
        ).otherwise(0)
    ).alias("customer_id_manquant"),
    F.sum(
        F.when(
            F.col("order_date").isNull()
            | (F.trim(F.col("order_date"))==""),
            1
        ).otherwise(0)
    ).alias("order_date_manquant"),
    F.sum(
        F.when(
            F.col("status").isNull()
            | (F.trim(F.col("status"))==""),
            1
        ).otherwise(0)
    ).alias("status_manquant"),
    F.sum(
        F.when(
            F.col("promotion_code").isNull()
            | (F.trim(F.col("promotion_code"))==""),
            1
        ).otherwise(0)
    ).alias("promotion_code_manquant")

).show()

+-----------+--------------------+-------------------+---------------+-----------------------+
|id_manquant|customer_id_manquant|order_date_manquant|status_manquant|promotion_code_manquant|
+-----------+--------------------+-------------------+---------------+-----------------------+
|          0|                   0|                  0|              0|                      9|
+-----------+--------------------+-------------------+---------------+-----------------------+



**-> We have 9 missing sales codes**

**->Let's see if there is any duplications of order_id**

In [156]:
orders_dup_ids = (
    orders_raw.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)
orders_dup_ids.show()

+--------+-----+
|order_id|count|
+--------+-----+
|   O0003|    2|
+--------+-----+



In [157]:
order_columns = orders_raw.columns

orders_duplications = (
    orders_raw.groupBy(order_columns)
    .agg(
        F.count("*").alias("occurences")
    )
    .filter(
        F.col("occurences") > 1
    )
    .withColumn("copie_key", F.col("order_id"))
    .withColumn("extra_copies", F.col("occurences") - 1)
    .withColumn("run_id", F.lit(run_id))
    
)
orders_duplications.show()

+--------+-----------+----------+---------+--------------+----------+---------+------------+--------------------+
|order_id|customer_id|order_date|   status|promotion_code|occurences|copie_key|extra_copies|              run_id|
+--------+-----------+----------+---------+--------------+----------+---------+------------+--------------------+
|   O0003|       C021|2025-05-10|CANCELLED|      PROMO10 |         2|    O0003|           1|f206b212-0ef8-41d...|
+--------+-----------+----------+---------+--------------+----------+---------+------------+--------------------+



**-> Let's check if the customers we have in orders table really exists in the silver**

In [158]:
customers_silver = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .parquet(f"{SILVER_DIR}/customers")
)

customers_silver.show(100, truncate=False)
print("nombre de customers silver: ",customers_silver.count())

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C020       |Youssef 20   |Ouest |2024-05-20       |
|C028       |Omar 28      |Centre|2024-07-15       |
|C011       |Nadia 11     |Nord  |2024-03-18       |
|C008       |Youssef 08   |Centre|2024-02-26       |
|C015       |Sara 15      |Ouest |2024-04-15       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C012       |Amine 12     |Sud   |2024-03-25       |
|C016       |Omar 16      |Nord  |2024-04-22       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
|C009       |Sara 09      |Est   |2024-03-04       |
|C007       |Alice 07     |Sud   |2024-02-19       |
|C006       |Amine 06     |Nord  |2024-02-12       |
|C029       |Nadia 29     |Est   |2024-07-22       |
|C030       |Amine 30     |Ouest |2024-07-29       |
|C010       |Omar 10      |Ouest |2024-03-11       |
|C017       |Nadia 17     |Sud   |2024-04-29  

In [159]:
orphans_customers = orders_raw.join(
    customers_silver.select("customer_id").distinct(),
    on="customer_id",
    how="left_anti"
)

orphans_customers.show(truncate=False)

+-----------+--------+----------+------+--------------+
|customer_id|order_id|order_date|status|promotion_code|
+-----------+--------+----------+------+--------------+
|C999       |O9001   |2025-06-20|PAID  |              |
+-----------+--------+----------+------+--------------+



**-> On a un seul customer id qui exist dans orders mais pas dans customers silver**

**-> Let's see the validation of "order_date"**

In [160]:
orders_typed = orders_raw.withColumn(
    "order_date_parsed",
    F.to_date("order_date","yyyy-MM-dd")
)
orders_typed.select(
    "order_id",
    "order_date",
    "order_date_parsed"
).show(100,truncate=False)

+--------+----------+-----------------+
|order_id|order_date|order_date_parsed|
+--------+----------+-----------------+
|O0001   |2025-05-04|2025-05-04       |
|O0002   |2025-05-07|2025-05-07       |
|O0003   |2025-05-10|2025-05-10       |
|O0004   |2025-05-13|2025-05-13       |
|O0005   |2025-05-16|2025-05-16       |
|O0006   |2025-05-19|2025-05-19       |
|O0007   |2025-05-22|2025-05-22       |
|O0008   |2025-05-25|2025-05-25       |
|O0009   |2025-05-28|2025-05-28       |
|O0010   |2025-05-31|2025-05-31       |
|O0011   |2025-06-03|2025-06-03       |
|O0012   |2025-06-06|2025-06-06       |
|O0013   |2025-06-09|2025-06-09       |
|O0014   |2025-06-12|2025-06-12       |
|O0015   |2025-06-15|2025-06-15       |
|O0016   |2025-06-18|2025-06-18       |
|O0017   |2025-06-21|2025-06-21       |
|O0018   |2025-06-24|2025-06-24       |
|O0019   |2025-06-27|2025-06-27       |
|O0020   |2025-06-30|2025-06-30       |
|O0021   |2025-07-03|2025-07-03       |
|O0022   |2025-07-06|2025-07-06       |


**-> The row with order_id "O9002" its parsed order_date is NULL**

**-> Let's verify the status of orders if they are the same as in known_status**

In [161]:
known_status = ["PAID", "DELIVERED", "CANCELLED" , "PENDING"]

orders_w_unk_status = orders_typed.filter(~F.col("status").isin(known_status))

orders_w_unk_status.show()

+--------+-----------+----------+-----------+--------------+-----------------+
|order_id|customer_id|order_date|     status|promotion_code|order_date_parsed|
+--------+-----------+----------+-----------+--------------+-----------------+
|   O9003|       C002|2025-06-20|IN_PROGRESS|          NULL|       2025-06-20|
+--------+-----------+----------+-----------+--------------+-----------------+



**-> Let's see if the sale codes are valid or no**

In [162]:
promotions_silver = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .parquet(f"{SILVER_DIR}/promotions")
)

promotions_silver.show(truncate=False)

+--------------+-------------+----------+----------+
|promotion_code|discount_rate|start_date|end_date  |
+--------------+-------------+----------+----------+
|PROMO5        |0.0500       |2025-01-01|2025-12-31|
|PROMO10       |0.1000       |2025-05-01|2025-05-31|
|SUMMER15      |0.1500       |2025-06-01|2025-08-31|
|VIP20         |0.2000       |2025-01-01|2025-12-31|
|OLD30         |0.3000       |2024-01-01|2024-12-31|
+--------------+-------------+----------+----------+



In [163]:
orders_typed = (
    orders_typed
    .withColumn("promotion_code", F.trim(F.col("promotion_code")))
)

In [164]:
orphan_promotions = orders_typed.join(
    promotions_silver.select("promotion_code").distinct(),
    on="promotion_code",
    how="left_anti"
)
orphan_promotions.show()

+--------------+--------+-----------+----------+-----------+-----------------+
|promotion_code|order_id|customer_id|order_date|     status|order_date_parsed|
+--------------+--------+-----------+----------+-----------+-----------------+
|              |   O0001|       C007|2025-05-04|       PAID|       2025-05-04|
|       UNKNOWN|   O0006|       C012|2025-05-19|  DELIVERED|       2025-05-19|
|              |   O0008|       C026|2025-05-25|    PENDING|       2025-05-25|
|       UNKNOWN|   O0013|       C001|2025-06-09|       PAID|       2025-06-09|
|              |   O0015|       C015|2025-06-15|  CANCELLED|       2025-06-15|
|       UNKNOWN|   O0020|       C020|2025-06-30|    PENDING|       2025-06-30|
|              |   O0022|       C004|2025-07-06|  DELIVERED|       2025-07-06|
|       UNKNOWN|   O0027|       C009|2025-07-21|  CANCELLED|       2025-07-21|
|              |   O0029|       C023|2025-07-27|       PAID|       2025-07-27|
|       UNKNOWN|   O0034|       C028|2025-05-03|  DE

In [165]:
orphan_promotions.groupBy("promotion_code").count().orderBy("count", ascending=False).show()

+--------------+-----+
|promotion_code|count|
+--------------+-----+
|              |    8|
|       UNKNOWN|    5|
|          NULL|    1|
+--------------+-----+



In [288]:
missing_status = F.col("status").isNull() | (F.trim(F.col("status"))=="")
unknown_status= ~missing_status & ~ F.col("status").isin(known_status)

orphan_customer_ids = [r["order_id"] for r in orphans_customers.select("order_id").collect()]


orders_checked = (
    orders_typed.withColumn(
        "quality_reason",
        F.concat_ws(
            " | ",
            F.when(
                F.col("order_id").isNull()
                | (F.trim(F.col("order_id"))==""),
                F.lit("ORDER_ID_MANQUANT")
            ),
            F.when(
                F.col("customer_id").isNull()
                | (F.trim(F.col("customer_id"))==""),
                F.lit("CUSTOMER_ID_MANQUANT")
            ),
            F.when(
                F.col("order_date").isNotNull()
                & (F.trim(F.col("order_date"))!="")
                & F.col("order_date_parsed").isNull(),
                F.lit("ORDER_DATE_INVALID")
            ),
            F.when(
                missing_status,
                F.lit("STATUS_MANQUANT")
            ),
            F.when(
                unknown_status,
                F.lit("STATUS_INVALID")
            ),
            F.when(
                F.col("order_id").isin(orphan_customer_ids),
                F.lit("CUSTOMER_ID_UNKNOWN")
            )
        )
    )
    
)

orders_checked.show(100, truncate=False)

+--------+-----------+----------+-----------+--------------+-----------------+-------------------+
|order_id|customer_id|order_date|status     |promotion_code|order_date_parsed|quality_reason     |
+--------+-----------+----------+-----------+--------------+-----------------+-------------------+
|O0001   |C007       |2025-05-04|PAID       |              |2025-05-04       |                   |
|O0002   |C014       |2025-05-07|DELIVERED  |PROMO5        |2025-05-07       |                   |
|O0003   |C021       |2025-05-10|CANCELLED  |PROMO10       |2025-05-10       |                   |
|O0004   |C028       |2025-05-13|PENDING    |SUMMER15      |2025-05-13       |                   |
|O0005   |C005       |2025-05-16|PAID       |VIP20         |2025-05-16       |                   |
|O0006   |C012       |2025-05-19|DELIVERED  |UNKNOWN       |2025-05-19       |                   |
|O0007   |C019       |2025-05-22|CANCELLED  |OLD30         |2025-05-22       |                   |
|O0008   |

In [289]:
orders_valid = (
    orders_checked.filter(
        (F.trim(F.col("quality_reason"))=="")
    )
    .dropDuplicates()
    .drop("order_date","quality_reason")
    .withColumnRenamed("order_date_parsed","order_date")
)

orders_valid.show(100,truncate=False)

+--------+-----------+---------+--------------+----------+
|order_id|customer_id|status   |promotion_code|order_date|
+--------+-----------+---------+--------------+----------+
|O0005   |C005       |PAID     |VIP20         |2025-05-16|
|O0004   |C028       |PENDING  |SUMMER15      |2025-05-13|
|O0026   |C002       |DELIVERED|VIP20         |2025-07-18|
|O0002   |C014       |DELIVERED|PROMO5        |2025-05-07|
|O0035   |C005       |CANCELLED|OLD30         |2025-05-06|
|O0019   |C013       |CANCELLED|VIP20         |2025-06-27|
|O0023   |C011       |CANCELLED|PROMO5        |2025-07-09|
|O0003   |C021       |CANCELLED|PROMO10       |2025-05-10|
|O0009   |C003       |PAID     |PROMO5        |2025-05-28|
|O0012   |C024       |PENDING  |VIP20         |2025-06-06|
|O0030   |C030       |DELIVERED|PROMO5        |2025-07-30|
|O0025   |C025       |PAID     |SUMMER15      |2025-07-15|
|O0036   |C012       |PENDING  |              |2025-05-09|
|O0029   |C023       |PAID     |              |2025-07-2

In [290]:
orders_quarantine = (
    orders_checked
    .filter(
        F.col("quality_reason").isNotNull() 
        & (F.trim(F.col("quality_reason"))!="")
    )
)

orders_quarantine.show()

+--------+-----------+----------+-----------+--------------+-----------------+-------------------+
|order_id|customer_id|order_date|     status|promotion_code|order_date_parsed|     quality_reason|
+--------+-----------+----------+-----------+--------------+-----------------+-------------------+
|   O9001|       C999|2025-06-20|       PAID|              |       2025-06-20|CUSTOMER_ID_UNKNOWN|
|   O9002|       C001|2025-02-30|       PAID|              |             NULL| ORDER_DATE_INVALID|
|   O9003|       C002|2025-06-20|IN_PROGRESS|          NULL|       2025-06-20|     STATUS_INVALID|
+--------+-----------+----------+-----------+--------------+-----------------+-------------------+



In [291]:
orders_valid.write.mode("overwrite").parquet(f"{SILVER_DIR}/orders")

orders_quarantine.write.mode("overwrite").parquet(f"{QUAR}/orders")

orders_duplications.write.mode("overwrite").parquet(f"{AUDIT_DIR}/orders_duplicates")

### **-> order_items**

In [292]:
order_items_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .parquet(f"{BRONZE_PQ}/order_items")
) 

order_items_raw.show(truncate=False)
order_items_raw.printSchema()
print("nombre de lignes: ", order_items_raw.count())

+-------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|quantity|unit_price|
+-------------+--------+----------+--------+----------+
|I0001        |O0001   |P003      |2       |200.00    |
|I0002        |O0001   |P008      |3       |180.00    |
|I0003        |O0002   |P006      |3       |75.00     |
|I0004        |O0002   |P011      |4       |65.00     |
|I0005        |O0003   |P009      |4       |95.00     |
|I0006        |O0003   |P002      |1       |20.00     |
|I0007        |O0004   |P012      |1       |25.00     |
|I0008        |O0004   |P005      |2       |320.00    |
|I0009        |O0005   |P003      |2       |190.00    |
|I0010        |O0005   |P008      |3       |180.00    |
|I0011        |O0006   |P006      |3       |75.00     |
|I0012        |O0006   |P011      |4       |65.00     |
|I0013        |O0007   |P009      |4       |95.00     |
|I0014        |O0007   |P002      |1       |20.00     |
|I0015        |O0008   |P012      |1       |25.0

**Lets check if there is any Null values**

In [293]:
order_items_raw.select(
    F.sum(
        F.when(
            F.col("order_item_id").isNull()
            | (F.trim(F.col("order_item_id"))==""),
            1
        ).otherwise(0)
    ).alias("ORDER_ITEM_ID_MANQUANT"),
    F.sum(
        F.when(
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id"))==""),
            1
        ).otherwise(0)
    ).alias("ORDER_ID_MANQUANT"),
    F.sum(
        F.when(
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id"))==""),
            1
        ).otherwise(0)
    ).alias("PRODUCT_ID_MANQUANT"),
    F.sum(
        F.when(
            F.col("quantity").isNull()
            | (F.trim(F.col("quantity"))==""),
            1
        ).otherwise(0)
    ).alias("QUANTITY_MANQUANT"),
    F.sum(
        F.when(
            F.col("unit_price").isNull()
            | (F.trim(F.col("unit_price"))==""),
            1
        ).otherwise(0)
    ).alias("UNIT_PRICE_MANQUANT"),
).show()

+----------------------+-----------------+-------------------+-----------------+-------------------+
|ORDER_ITEM_ID_MANQUANT|ORDER_ID_MANQUANT|PRODUCT_ID_MANQUANT|QUANTITY_MANQUANT|UNIT_PRICE_MANQUANT|
+----------------------+-----------------+-------------------+-----------------+-------------------+
|                     0|                0|                  0|                0|                  0|
+----------------------+-----------------+-------------------+-----------------+-------------------+



In [294]:
dup_items_ids = (
    order_items_raw.groupBy("order_item_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)
dup_items_ids.show()

+-------------+-----+
|order_item_id|count|
+-------------+-----+
|        I0004|    2|
+-------------+-----+



In [295]:
order_item_column = order_items_raw.columns

order_item_duplications = (
    order_items_raw.groupBy(order_item_column)
    .agg(
        F.count("*").alias("occurences")
    )
    .filter(
        F.col("occurences") > 1
    )
    .withColumn("copie_key", F.col("order_item_id"))
    .withColumn("extra_copies", F.col("occurences") - 1)
    .withColumn("run_id", F.lit(run_id))
)
order_item_duplications.show()

+-------------+--------+----------+--------+----------+----------+---------+------------+--------------------+
|order_item_id|order_id|product_id|quantity|unit_price|occurences|copie_key|extra_copies|              run_id|
+-------------+--------+----------+--------+----------+----------+---------+------------+--------------------+
|        I0004|   O0002|      P011|       4|    65.00 |         2|    I0004|           1|f206b212-0ef8-41d...|
+-------------+--------+----------+--------+----------+----------+---------+------------+--------------------+



In [296]:
silver_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .parquet(f"{SILVER_DIR}/orders")
)
silver_orders.show(5)

silver_products=(
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .parquet(f"{SILVER_DIR}/products")
)
silver_products.show(5)

+--------+-----------+---------+--------------+----------+
|order_id|customer_id|   status|promotion_code|order_date|
+--------+-----------+---------+--------------+----------+
|   O0005|       C005|     PAID|         VIP20|2025-05-16|
|   O0004|       C028|  PENDING|      SUMMER15|2025-05-13|
|   O0026|       C002|DELIVERED|         VIP20|2025-07-18|
|   O0002|       C014|DELIVERED|        PROMO5|2025-05-07|
|   O0035|       C005|CANCELLED|         OLD30|2025-05-06|
+--------+-----------+---------+--------------+----------+
only showing top 5 rows

+----------+------------+------------+------------------+---------+
|product_id|product_name|    category|unit_catalog_price|unit_cost|
+----------+------------+------------+------------------+---------+
|      P011|  Microphone|       Audio|             65.00|    35.00|
|      P001|      Laptop|Informatique|            800.00|   600.00|
|      P010|      Router|      Réseau|            110.00|    70.00|
|      P005|      Tablet|Informatiqu

In [297]:
orphan_products= order_items_raw.join(
    silver_products.select("product_id").distinct(),
    on="product_id",
    how="left_anti"
)
orphan_products.show()

orphan_orders = order_items_raw.join(
    silver_orders.select("order_id").distinct(),
    on="order_id",
    how="left_anti"
)
orphan_orders.show()


+----------+-------------+--------+--------+----------+
|product_id|order_item_id|order_id|quantity|unit_price|
+----------+-------------+--------+--------+----------+
|      P999|        I9002|   O0001|       2|    10.00 |
+----------+-------------+--------+--------+----------+

+--------+-------------+----------+--------+----------+
|order_id|order_item_id|product_id|quantity|unit_price|
+--------+-------------+----------+--------+----------+
|   O9999|        I9001|      P001|       1|   800.00 |
+--------+-------------+----------+--------+----------+



**-> Let's parse the types of unit_price and quantity**

In [298]:
from pyspark.sql.types import IntegerType

In [299]:
order_item_typed = (
    order_items_raw.withColumn(
        "quantity",
        F.col("quantity").cast(IntegerType())
    )
)

order_item_typed = (
    order_item_typed.withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(10,2)")
    )
)

order_item_typed.printSchema()

root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)



In [300]:
negative_quantity = (
    order_item_typed.filter(
        F.col("quantity") < 0
    )
)
negative_quantity.show()

+-------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|quantity|unit_price|
+-------------+--------+----------+--------+----------+
|        I9003|   O0002|      P002|      -2|     20.00|
+-------------+--------+----------+--------+----------+



In [301]:
negative_unit_price = (
    order_item_typed.filter(
        (F.col("unit_price") < 0)
    )
)
negative_unit_price.show()

+-------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|quantity|unit_price|
+-------------+--------+----------+--------+----------+
+-------------+--------+----------+--------+----------+



In [302]:
product_orphans_ids = [r["product_id"] for r in orphan_products.collect()]

order_orphan_ids = [r["order_id"] for r in orphan_orders.collect()]

order_item_checked = (
    order_item_typed.withColumn(
        "quality_reason",
        F.concat_ws(
            " | ",
            F.when(
                F.col("order_item_id").isNull()
                | (F.trim(F.col("order_item_id"))==""),
                F.lit("ORDER_ITEM_ID_MANQUANT")
            ),
            F.when(
                F.col("order_id").isNull()
                | (F.trim(F.col("order_id"))==""),
                F.lit("ORDER_ID_MANQUANT")
            ),
            F.when(
                F.col("order_id").isin(order_orphan_ids),
                F.lit("ORDER_ID_UNKNOWN")
            ),
            F.when(
                F.col("product_id").isNull()
                | (F.trim(F.col("product_id"))==""),
                F.lit("PRODUCT_ID_MANQUANT")
            ),
            F.when(
                F.col("product_id").isin(product_orphans_ids),
                F.lit("PRODUCT_ID_UNKNOWN")
            ),

            F.when(
                F.col("quantity").isNull()
                | (F.trim(F.col("quantity"))==""),
                F.lit("QUANTITY_MANQUANT")
            ),
            F.when(
                (F.col("quantity").isNotNull())
                & (F.col("quantity")<0),
                F.lit("QUANTITY_NEGATIVE")
            ),
            F.when(
                F.col("unit_price").isNull()
                | (F.trim(F.col("unit_price"))==""),
                F.lit("UNIT_PRICE_MANQUANT")
            )

        )
    )
)

order_item_checked.show(100,truncate=False)

+-------------+--------+----------+--------+----------+------------------+
|order_item_id|order_id|product_id|quantity|unit_price|quality_reason    |
+-------------+--------+----------+--------+----------+------------------+
|I0001        |O0001   |P003      |2       |200.00    |                  |
|I0002        |O0001   |P008      |3       |180.00    |                  |
|I0003        |O0002   |P006      |3       |75.00     |                  |
|I0004        |O0002   |P011      |4       |65.00     |                  |
|I0005        |O0003   |P009      |4       |95.00     |                  |
|I0006        |O0003   |P002      |1       |20.00     |                  |
|I0007        |O0004   |P012      |1       |25.00     |                  |
|I0008        |O0004   |P005      |2       |320.00    |                  |
|I0009        |O0005   |P003      |2       |190.00    |                  |
|I0010        |O0005   |P008      |3       |180.00    |                  |
|I0011        |O0006   |P

In [303]:
order_item_valid = (
    order_item_checked.filter(
        (F.trim(F.col("quality_reason"))=="")
    )
    .dropDuplicates()
    .drop("quality_reason")
    .withColumnRenamed("unit_price","transaction_unit_price")
)
order_item_valid.show(100, truncate=False)


+-------------+--------+----------+--------+----------------------+
|order_item_id|order_id|product_id|quantity|transaction_unit_price|
+-------------+--------+----------+--------+----------------------+
|I0066        |O0033   |P008      |3       |180.00                |
|I0063        |O0032   |P012      |1       |23.75                 |
|I0001        |O0001   |P003      |2       |200.00                |
|I0029        |O0015   |P009      |4       |95.00                 |
|I0009        |O0005   |P003      |2       |190.00                |
|I0039        |O0020   |P012      |1       |25.00                 |
|I0067        |O0034   |P006      |3       |75.00                 |
|I0079        |O0040   |P012      |1       |25.00                 |
|I0026        |O0013   |P008      |3       |180.00                |
|I0058        |O0029   |P008      |3       |180.00                |
|I0059        |O0030   |P006      |3       |75.00                 |
|I0070        |O0035   |P002      |1       |20.0

In [304]:
order_item_quarantine = (
    order_item_checked.filter(
        (F.trim(F.col("quality_reason"))!="")
    )
)
order_item_quarantine.show()

+-------------+--------+----------+--------+----------+------------------+
|order_item_id|order_id|product_id|quantity|unit_price|    quality_reason|
+-------------+--------+----------+--------+----------+------------------+
|        I9001|   O9999|      P001|       1|    800.00|  ORDER_ID_UNKNOWN|
|        I9002|   O0001|      P999|       2|     10.00|PRODUCT_ID_UNKNOWN|
|        I9003|   O0002|      P002|      -2|     20.00| QUANTITY_NEGATIVE|
|        I9004|   O0004|      P004|    NULL|     45.00| QUANTITY_MANQUANT|
+-------------+--------+----------+--------+----------+------------------+



In [305]:
order_item_valid.write.mode("overwrite").parquet(f"{SILVER_DIR}/order_items")
order_item_quarantine.write.mode("overwrite").parquet(f"{QUAR}/order_items")
order_item_duplications.write.mode("overwrite").parquet(f"{AUDIT_DIR}/order_items_duplicates")

### **Tableaux de synthèse qualité** 

**-> Functions to simplify:**

In [306]:
def count_conflicts( df, key, data_cols=None):
    cols= data_cols or df.columns
    conflicts = (
        df
        .groupBy(key)
        .count()
        .filter(
            F.col("count") > 1
        )
    )
    
    n_ids = conflicts.count()
    return n_ids


def count_rows(df):
    return df.count()

def delete_reasons(df):
    reasons = (
        df
        .select(F.explode(F.split("quality_reason",r" \| ")).alias("reason"))
        .filter((F.trim(F.col("reason"))!=""))
        .agg(
            F.concat_ws(", ", F.sort_array(F.collect_set("reason"))).alias("distinct_reasons")
        )
    )
    return reasons



def conservation(name, bronze_df, silver_df, quarantine_df, dup_audit_df):
    n_initial = count_rows(bronze_df)
    n_valid   = count_rows(silver_df)
    n_quar    = count_rows(quarantine_df)
    n_extra   = count_rows(dup_audit_df)
    
    ok = (n_initial == n_valid + n_quar + n_extra)
    return (name, n_valid + n_quar + int(n_extra),
            "Oui" if ok else "Non")


In [307]:
from functools import reduce
from pyspark.sql.types import LongType


In [308]:

# Schéma défini une seule fois, réutilisé par chaque appel
SCHEMA_SYNTHESE = StructType([
    StructField("Source", StringType(), True),
    StructField("Nombre de lignes initiales", LongType(), True),
    StructField("Nombre d'enregistrements canoniques valides", LongType(), True),
    StructField("Nombre de lignes invalides en quarantaine", LongType(), True),
    StructField("Nombre de copies supplémentaires tracées", LongType(), True),
    StructField("Nombre de conflits d'identifiants", LongType(), True),
    StructField("Nombre de références inexistantes", LongType(), True),  # nullable : N/A
    StructField("Principaux motifs de rejet", StringType(), True),
    StructField("Conservation des lignes vérifiée", StringType(), True),
])


def _compter_copies_supplementaires(dup_audit_df):
    """Somme des copies supplémentaires (et non le nombre de groupes)."""
    if dup_audit_df is None:
        return 0
    return dup_audit_df.agg(F.sum("extra_copies")).first()[0] or 0


def _motifs_de_rejet(quarantine_df):
    """Une seule chaîne triée, calculée sur la quarantaine."""
    return (
        quarantine_df
        .select(F.explode(F.split("quality_reason", r" \| ")).alias("r"))
        .filter("r != ''")
        .agg(F.concat_ws(", ", F.sort_array(F.collect_set("r"))))
        .first()[0] or "Aucun"
    )


def table_de_synthese(nom, key, bronze_df, inter_df, valid_df, quarantine_df,
                      dup_audit_df, orphans_df=None):
    n_initial  = bronze_df.count()
    n_valid    = valid_df.count()
    n_quar     = quarantine_df.count()
    n_extra    = int(_compter_copies_supplementaires(dup_audit_df))
    n_conflict = count_conflicts(inter_df, key)  # adapter à la signature de ta fonction
    n_orphans  = orphans_df.count() if orphans_df is not None else None  # None = N/A

    reasons = _motifs_de_rejet(quarantine_df)
    ok = "Oui" if n_initial == n_valid + n_quar + n_extra else "Non"

    row = [(
        nom,
        int(n_initial),
        int(n_valid),
        int(n_quar),
        n_extra,
        int(n_conflict),
        0 if n_orphans is None else int(n_orphans),
        reasons,
        ok,
    )]

    return spark.createDataFrame(row, SCHEMA_SYNTHESE)

**-> Customers**

In [309]:
from functools import reduce

synth_customers = table_de_synthese("customers", "customer_id",
    customers_raw, customers_typed, customers_valid, customers_quarantine, customers_duplications)

synth_products = table_de_synthese("products", "product_id",
    products_raw, products_typed, products_valid, products_quarantine, products_duplications)

synth_orders = table_de_synthese("orders", "order_id",
    orders_raw, orders_typed, orders_valid, orders_quarantine, orders_duplications,
    orphans_df=orphans_customers)           # clients inexistants

synth_items = table_de_synthese("order_items", "order_item_id",
    order_items_raw, order_item_typed, order_item_valid, order_item_quarantine,
    order_item_duplications,
    orphans_df= orphan_orders)      # commandes + produits inexistants

synth_promos = table_de_synthese("promotions", "promotion_code",
    promotions_raw, promotions_typed, promotions_silver, promotions_quarantine,
    dup_audit_df=None)                   # pas de doublons exacts

synthese = reduce(lambda a, b: a.unionByName(b),
                  [synth_customers, synth_products, synth_orders, synth_items, synth_promos])

synthese.show(truncate=False)

+-----------+--------------------------+-------------------------------------------+-----------------------------------------+----------------------------------------+---------------------------------+---------------------------------+--------------------------------------------------------------------------------------+--------------------------------+
|Source     |Nombre de lignes initiales|Nombre d'enregistrements canoniques valides|Nombre de lignes invalides en quarantaine|Nombre de copies supplémentaires tracées|Nombre de conflits d'identifiants|Nombre de références inexistantes|Principaux motifs de rejet                                                            |Conservation des lignes vérifiée|
+-----------+--------------------------+-------------------------------------------+-----------------------------------------+----------------------------------------+---------------------------------+---------------------------------+-----------------------------------------------------

## **Étape 4 : Jointures et construction de la vue commerciale**

**order_items x orders**

In [310]:
orders_silver = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .parquet(f"{SILVER_DIR}/orders")
)
order_items_silver= (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .parquet(f"{SILVER_DIR}/order_items")
)
products_silver= (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .parquet(f"{SILVER_DIR}/products")
)

**1.**

**->Check unicité**

In [311]:
def check_unique(df, key, nom):
    total = df.count()
    distincts = df.select(key).distinct().count()
    print(f"{nom}: {total} lignes, {distincts} clés distinctes -> {'OK' if total == distincts else 'DUPLICATAS'}")

check_unique(customers_silver,"customer_id","customers")
check_unique(orders_silver,"order_id","orders")
check_unique(products_silver,"product_id","products")
check_unique(promotions_silver,"promotion_code","promotions")
check_unique(order_items_silver,"order_item_id","order_items")

customers: 30 lignes, 30 clés distinctes -> OK
orders: 40 lignes, 40 clés distinctes -> OK
products: 12 lignes, 12 clés distinctes -> OK
promotions: 5 lignes, 5 clés distinctes -> OK
order_items: 81 lignes, 81 clés distinctes -> OK


**-> Validité des promotions pour les commandes**

In [ ]:
customers = customers_silver.select("customer_id" , "customer_name","region")

sales_view = (
    order_items_silver.join(orders_silver, "order_id","inner")
    .join(products_silver,"product_id","inner")
    .join(customers,"customer_id","inner")
    .join(promotions_silver,"promotion_code","left")
    .withColumn("is_eligible", F.col("status").isin("PAID","DELIVERED"))
    .withColumn(
    "is_promo_valid",
        F.when(
            (F.col("order_date") >= F.col("start_date")) 
            & (F.col("order_date") <= F.col("end_date")) ,
            F.lit("PROMO_VALID")
        ).otherwise(F.lit("PROMO_INVALID")),
        F.when
    )
    .withColumnRenamed("start_date","promo_start_date")
    .withColumnRenamed("end_date","promo_end_date")
)

sales_view = sales_view.select(
    # Identifiants (traçabilité)
    "order_item_id", "order_id", "customer_id", "product_id",

    # Commande
    "order_date", "status", "is_eligible",

    # Client
    "customer_name", "region", 

    # Produit
    "product_name", "category", "unit_catalog_price", "unit_cost",

    # Transaction
    "quantity", "transaction_unit_price",

    # Promotion
    "promotion_code", "discount_rate", "promo_start_date", "promo_end_date",
    "is_promo_valid",

    
)
sales_view.show(truncate=False)

+-------------+--------+-----------+----------+----------+---------+-----------+-------------+------+------------+------------+------------------+---------+--------+----------------------+--------------+-------------+----------------+--------------+--------------+
|order_item_id|order_id|customer_id|product_id|order_date|status   |is_eligible|customer_name|region|product_name|category    |unit_catalog_price|unit_cost|quantity|transaction_unit_price|promotion_code|discount_rate|promo_start_date|promo_end_date|is_promo_valid|
+-------------+--------+-----------+----------+----------+---------+-----------+-------------+------+------------+------------+------------------+---------+--------+----------------------+--------------+-------------+----------------+--------------+--------------+
|I0066        |O0033   |C021       |P008      |2025-08-08|PAID     |true       |Sara 21      |Nord  |Printer     |Bureau      |180.00            |125.00   |3       |180.00                |VIP20         |0.

In [313]:
print("order_items avant: ",order_items_silver.count())

print("order_items aprés: ",sales_view.count())



order_items avant:  81


order_items aprés:  81


In [314]:

# 1. Lignes de order_items_silver absentes de sales_view
excluded = order_items_silver.join(
    sales_view.select("order_item_id","status"), "order_item_id", "left_anti")

# 2. Motif : la commande et/ou le produit n'existent pas dans Silver
order_ids   = orders_silver.select("order_id").distinct()      .withColumn("order_ok", F.lit(True))
product_ids = products_silver.select("product_id").distinct()  .withColumn("product_ok", F.lit(True))

excluded_reasons = (
    excluded
    .join(order_ids,   "order_id",   "left")
    .join(product_ids, "product_id", "left")
    .withColumn("exclusion_reason",
        F.concat_ws(" | ",
            F.when(F.col("order_ok").isNull(),   F.lit("COMMANDE_ABSENTE_DE_SILVER")),
            F.when(F.col("product_ok").isNull(), F.lit("PRODUIT_ABSENT_DE_SILVER"))))
)

excluded_reasons.show(truncate=False)
excluded_reasons.groupBy("exclusion_reason").count().show(truncate=False)

+----------+--------+-------------+--------+----------------------+--------+----------+----------------+
|product_id|order_id|order_item_id|quantity|transaction_unit_price|order_ok|product_ok|exclusion_reason|
+----------+--------+-------------+--------+----------------------+--------+----------+----------------+
+----------+--------+-------------+--------+----------------------+--------+----------+----------------+

+----------------+-----+
|exclusion_reason|count|
+----------------+-----+
+----------------+-----+



In [315]:
excluded_orders_ids = [r["order_id"] for r in excluded.select("order_id").distinct().collect()]

excluded_orders_status = (
    orders_silver
    .filter(
        F.col("order_id").isin(excluded_orders_ids)
    )
    .select("promotion_code")
)
excluded_orders_status.show()
print(excluded_orders_ids)

+--------------+
|promotion_code|
+--------------+
+--------------+

[]


In [316]:
orphan_orders.show()

+--------+-------------+----------+--------+----------+
|order_id|order_item_id|product_id|quantity|unit_price|
+--------+-------------+----------+--------+----------+
|   O9999|        I9001|      P001|       1|   800.00 |
+--------+-------------+----------+--------+----------+



In [317]:
orphan_orders_ids = [r["order_id"] for r in orphan_orders.collect()]

test = orders_silver.filter(
    F.col("order_id").isin(orphan_orders_ids)
)
test.show()

+--------+-----------+------+--------------+----------+
|order_id|customer_id|status|promotion_code|order_date|
+--------+-----------+------+--------------+----------+
+--------+-----------+------+--------------+----------+



In [318]:
orphan_products.show()

+----------+-------------+--------+--------+----------+
|product_id|order_item_id|order_id|quantity|unit_price|
+----------+-------------+--------+--------+----------+
|      P999|        I9002|   O0001|       2|    10.00 |
+----------+-------------+--------+--------+----------+



In [319]:
VIEW_PATH =f"{SILVER_DIR}/sales_view"
sales_view.write.mode("overwrite").parquet(VIEW_PATH)

In [320]:
sales_view.groupBy("is_eligible").count().show()

+-----------+-----+
|is_eligible|count|
+-----------+-----+
|       true|   41|
|      false|   40|
+-----------+-----+



In [321]:
spark.read.parquet(VIEW_PATH).count()==sales_view.count()

True

In [322]:
sales_view.printSchema()

root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- is_eligible: boolean (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_catalog_price: decimal(10,2) (nullable = true)
 |-- unit_cost: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_unit_price: decimal(10,2) (nullable = true)
 |-- promotion_code: string (nullable = true)
 |-- discount_rate: decimal(5,4) (nullable = true)
 |-- promo_start_date: date (nullable = true)
 |-- promo_end_date: date (nullable = true)
 |-- is_promo_valid: string (nullable = false)



## **Étape 5 : Zone Gold et indicateurs commerciaux**

#### **Construction de Gold**

In [323]:
gold_table = (
    spark.read
    .option("header",True)
    .option("inferSchema",True)
    .parquet(VIEW_PATH)
)
gold_table.printSchema()

root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- is_eligible: boolean (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_catalog_price: decimal(10,2) (nullable = true)
 |-- unit_cost: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_unit_price: decimal(10,2) (nullable = true)
 |-- promotion_code: string (nullable = true)
 |-- discount_rate: decimal(5,4) (nullable = true)
 |-- promo_start_date: date (nullable = true)
 |-- promo_end_date: date (nullable = true)
 |-- is_promo_valid: string (nullable = true)



In [324]:
gold_table.select("status").distinct().show()

+---------+
|   status|
+---------+
|DELIVERED|
|     PAID|
|CANCELLED|
|  PENDING|
+---------+



**-> Garde que les commandes eligibles**

In [325]:
gold_table = gold_table.filter(
    F.col("is_eligible")==True
)
gold_table= gold_table.drop("is_eligible")

gold_table.show(100,truncate=True)

+-------------+--------+-----------+----------+----------+---------+-------------+------+------------+------------+------------------+---------+--------+----------------------+--------------+-------------+----------------+--------------+--------------+
|order_item_id|order_id|customer_id|product_id|order_date|   status|customer_name|region|product_name|    category|unit_catalog_price|unit_cost|quantity|transaction_unit_price|promotion_code|discount_rate|promo_start_date|promo_end_date|is_promo_valid|
+-------------+--------+-----------+----------+----------+---------+-------------+------+------------+------------+------------------+---------+--------+----------------------+--------------+-------------+----------------+--------------+--------------+
|        I0066|   O0033|       C021|      P008|2025-08-08|     PAID|      Sara 21|  Nord|     Printer|      Bureau|            180.00|   125.00|       3|                180.00|         VIP20|       0.2000|      2025-01-01|    2025-12-31|   P

In [326]:
gold_table.agg(F.countDistinct("order_id").alias("commandes_eligibles")).show()

+-------------------+
|commandes_eligibles|
+-------------------+
|                 20|
+-------------------+



In [327]:
gold_table = (
    gold_table.withColumn(
        "new_discount_rate",
        F.when(
            (F.trim(F.col("is_promo_valid"))=="PROMO_INVALID"),
            F.lit(0).cast("decimal(5,4)")
        ).otherwise(F.col("discount_rate"))
    )
)
gold_table.select("is_promo_valid","discount_rate","new_discount_rate").show()

+--------------+-------------+-----------------+
|is_promo_valid|discount_rate|new_discount_rate|
+--------------+-------------+-----------------+
|   PROMO_VALID|       0.2000|           0.2000|
| PROMO_INVALID|         NULL|           0.0000|
|   PROMO_VALID|       0.2000|           0.2000|
| PROMO_INVALID|         NULL|           0.0000|
| PROMO_INVALID|         NULL|           0.0000|
| PROMO_INVALID|         NULL|           0.0000|
|   PROMO_VALID|       0.0500|           0.0500|
| PROMO_INVALID|         NULL|           0.0000|
|   PROMO_VALID|       0.2000|           0.2000|
| PROMO_INVALID|         NULL|           0.0000|
|   PROMO_VALID|       0.1000|           0.1000|
| PROMO_INVALID|       0.3000|           0.0000|
|   PROMO_VALID|       0.1500|           0.1500|
| PROMO_INVALID|         NULL|           0.0000|
|   PROMO_VALID|       0.1500|           0.1500|
|   PROMO_VALID|       0.0500|           0.0500|
| PROMO_INVALID|       0.3000|           0.0000|
|   PROMO_VALID|    

In [328]:
gold_table.printSchema()

root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_catalog_price: decimal(10,2) (nullable = true)
 |-- unit_cost: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_unit_price: decimal(10,2) (nullable = true)
 |-- promotion_code: string (nullable = true)
 |-- discount_rate: decimal(5,4) (nullable = true)
 |-- promo_start_date: date (nullable = true)
 |-- promo_end_date: date (nullable = true)
 |-- is_promo_valid: string (nullable = true)
 |-- new_discount_rate: decimal(5,4) (nullable = true)



In [329]:
gold_table=gold_table.drop("discount_rate")
gold_table=gold_table.withColumnRenamed("new_discount_rate","discount_rate")
gold_table.show()

+-------------+--------+-----------+----------+----------+---------+-------------+------+------------+------------+------------------+---------+--------+----------------------+--------------+----------------+--------------+--------------+-------------+
|order_item_id|order_id|customer_id|product_id|order_date|   status|customer_name|region|product_name|    category|unit_catalog_price|unit_cost|quantity|transaction_unit_price|promotion_code|promo_start_date|promo_end_date|is_promo_valid|discount_rate|
+-------------+--------+-----------+----------+----------+---------+-------------+------+------------+------------+------------------+---------+--------+----------------------+--------------+----------------+--------------+--------------+-------------+
|        I0066|   O0033|       C021|      P008|2025-08-08|     PAID|      Sara 21|  Nord|     Printer|      Bureau|            180.00|   125.00|       3|                180.00|         VIP20|      2025-01-01|    2025-12-31|   PROMO_VALID|   

**-> CA net**

In [330]:
gold_table = gold_table.withColumn(
    "CA_net",
    F.col("quantity") * F.col("transaction_unit_price") * ( 1 - F.col("discount_rate"))
)
gold_table.select("CA_net").show()



+----------+
|    CA_net|
+----------+
|432.000000|
|400.000000|
|304.000000|
|225.000000|
|540.000000|
|540.000000|
|213.750000|
|225.000000|
|180.000000|
|225.000000|
|202.500000|
|213.750000|
|459.000000|
|400.000000|
|191.250000|
|513.000000|
|400.000000|
|380.000000|
|260.000000|
|432.000000|
+----------+
only showing top 20 rows



**-> Cout total**

In [331]:
gold_table = gold_table.withColumn(
    "total_cost",
    F.col("quantity") * F.col("unit_cost")
)

gold_table.select("total_cost").show()

+----------+
|total_cost|
+----------+
|    375.00|
|    280.00|
|    280.00|
|    120.00|
|    375.00|
|    375.00|
|    120.00|
|    120.00|
|    120.00|
|    120.00|
|    120.00|
|    120.00|
|    375.00|
|    280.00|
|    120.00|
|    375.00|
|    280.00|
|    280.00|
|    140.00|
|    375.00|
+----------+
only showing top 20 rows



**-> Marge**

In [332]:
gold_table = gold_table.withColumn(
    "marge",
    F.col("CA_net") - F.col("total_cost")
)

gold_table.select("CA_net","total_cost","marge").show()

+----------+----------+----------+
|    CA_net|total_cost|     marge|
+----------+----------+----------+
|432.000000|    375.00| 57.000000|
|400.000000|    280.00|120.000000|
|304.000000|    280.00| 24.000000|
|225.000000|    120.00|105.000000|
|540.000000|    375.00|165.000000|
|540.000000|    375.00|165.000000|
|213.750000|    120.00| 93.750000|
|225.000000|    120.00|105.000000|
|180.000000|    120.00| 60.000000|
|225.000000|    120.00|105.000000|
|202.500000|    120.00| 82.500000|
|213.750000|    120.00| 93.750000|
|459.000000|    375.00| 84.000000|
|400.000000|    280.00|120.000000|
|191.250000|    120.00| 71.250000|
|513.000000|    375.00|138.000000|
|400.000000|    280.00|120.000000|
|380.000000|    280.00|100.000000|
|260.000000|    140.00|120.000000|
|432.000000|    375.00| 57.000000|
+----------+----------+----------+
only showing top 20 rows



In [333]:
gold_table.show()

+-------------+--------+-----------+----------+----------+---------+-------------+------+------------+------------+------------------+---------+--------+----------------------+--------------+----------------+--------------+--------------+-------------+----------+----------+----------+
|order_item_id|order_id|customer_id|product_id|order_date|   status|customer_name|region|product_name|    category|unit_catalog_price|unit_cost|quantity|transaction_unit_price|promotion_code|promo_start_date|promo_end_date|is_promo_valid|discount_rate|    CA_net|total_cost|     marge|
+-------------+--------+-----------+----------+----------+---------+-------------+------+------------+------------+------------------+---------+--------+----------------------+--------------+----------------+--------------+--------------+-------------+----------+----------+----------+
|        I0066|   O0033|       C021|      P008|2025-08-08|     PAID|      Sara 21|  Nord|     Printer|      Bureau|            180.00|   125.0

In [334]:
GOLD_PATH =f"{BASE_DIR}/data_lakehouse/gold"

gold_table.write.mode("overwrite").parquet(f"{GOLD_PATH}")

**Indicateurs globaux**

In [335]:
n_eligible_orders = gold_table.agg(
    F.countDistinct("order_id").alias("n_eligible_orders")
)
n_eligible_orders.show()

+-----------------+
|n_eligible_orders|
+-----------------+
|               20|
+-----------------+



In [336]:
n_gold_rows = gold_table.count()
print("le nombre de lignes Gold ",n_gold_rows)

le nombre de lignes Gold  41


In [337]:
n_customers = gold_table.agg(
    F.countDistinct("customer_id").alias("n_concerned_customers")
)
n_customers.show()

+---------------------+
|n_concerned_customers|
+---------------------+
|                   20|
+---------------------+



In [338]:
total_quantity = gold_table.agg(
    F.sum("quantity").alias("total_quantity")
)
total_quantity.show()

+--------------+
|total_quantity|
+--------------+
|           120|
+--------------+



In [339]:
gold_table.printSchema()

root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_catalog_price: decimal(10,2) (nullable = true)
 |-- unit_cost: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_unit_price: decimal(10,2) (nullable = true)
 |-- promotion_code: string (nullable = true)
 |-- promo_start_date: date (nullable = true)
 |-- promo_end_date: date (nullable = true)
 |-- is_promo_valid: string (nullable = true)
 |-- discount_rate: decimal(5,4) (nullable = true)
 |-- CA_net: decimal(28,6) (nullable = true)
 |-- total_cost: decimal(21,2) (nullable = true)
 |-- marge: decimal(29,6) (nullable = true)



In [347]:
from decimal import Decimal, ROUND_HALF_UP


In [348]:
ca_net = gold_table.agg(
    F.sum("CA_net").alias("CA_net")
).first()

total_cost = gold_table.agg(
    F.sum("total_cost").alias("total_cost")
).first()

total_marge= gold_table.agg(
    F.sum("marge").alias("total_marge")
).first()

def r2(x):
    return None if x is None else x.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

# tes valeurs (Decimal)
ca_net_v     = ca_net["CA_net"]
total_cost_v = total_cost["total_cost"]
marge_v      = total_marge["total_marge"]

# ratio avec gestion du dénominateur nul
taux_marge_v = (marge_v / ca_net_v * 100) if ca_net_v else None

kpi_table = spark.createDataFrame(
    [(r2(ca_net_v), r2(total_cost_v), r2(marge_v), r2(taux_marge_v))],
    "ca_net decimal(14,2), total_cost decimal(14,2), total_marge decimal(14,2), taux_marge decimal(10,2)"
)

kpi_table.show()


+--------+----------+-----------+----------+
|  ca_net|total_cost|total_marge|taux_marge|
+--------+----------+-----------+----------+
|13259.80|   9150.00|    4109.80|     30.99|
+--------+----------+-----------+----------+



In [352]:
avg_cart = (
    gold_table.agg(
        F.sum("CA_net").alias("ca_net"),
        F.countDistinct("order_id").alias("commandes_eligibles")
    )
    .withColumn(
        "panier_moyen",
        F.when(F.col("commandes_eligibles") == 0, F.lit(None).cast("decimal(14,2)"))
        .otherwise(F.round(F.col("ca_net") / F.col("commandes_eligibles"),2) )
    )
)

avg_cart.show()

+------------+-------------------+------------+
|      ca_net|commandes_eligibles|panier_moyen|
+------------+-------------------+------------+
|13259.800000|                 20|      662.99|
+------------+-------------------+------------+

